In [9]:
# E0: Native-language baseline
# XLM-RoBERTa fine-tuned directly on the low-resource dataset

import json
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


In [10]:
import sys
import torch

print("Python path:", sys.executable)
print("Torch path:", torch.__file__)
print("Torch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

Python path: /usr/bin/python3
Torch path: /home/jay/.local/lib/python3.10/site-packages/torch/__init__.py
Torch version: 2.5.1+cu121
CUDA Available: True


In [ ]:

# ============================================================
# 1. Configuration
# ============================================================

MODEL_NAME = "xlm-roberta-base"

TRAIN_DATA_PATH = "db/goemotions/train.tsv"
VALID_DATA_PATH = "db/goemotions/dev.tsv"
TEST_DATA_PATH = "db/goemotions/test.tsv"
EMOTIONS_PATH = "db/goemotions/emotions.txt"
EKMAN_PATH = "db/goemotions/ekman_mapping.json"

TEXT_COLUMN = "text"
LABEL_COLUMN = "emotion"

MAX_LENGTH = 140
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-4

RANDOM_SEED = 42

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)


Using device: cuda


In [13]:

# ============================================================
# 2. Reproducibility
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_SEED)



In [14]:

# ============================================================
# 3. Load dataset
# ============================================================

train_df = pd.read_csv(TRAIN_DATA_PATH, sep='\t', header=None, names=["text", "emotion", "userid"])

# Remove rows with missing text/labels
train_df = train_df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])

print("Number of examples:", len(train_df))
print("\nClass distribution:")
print(train_df[LABEL_COLUMN].value_counts())



Number of examples: 43410

Class distribution:
emotion
27           12823
0             2710
4             1873
15            1857
1             1652
             ...  
0,4,9,18         1
3,9,26           1
0,8,15,17        1
8,13,17          1
6,12,25          1
Name: count, Length: 711, dtype: int64


In [15]:
test_df = pd.read_csv(TEST_DATA_PATH, sep='\t', header=None, names=["text", "emotion", "userid"])

# Remove rows with missing text/labels
test_df = test_df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])

print("Number of examples:", len(test_df))
print("\nClass distribution:")
print(test_df[LABEL_COLUMN].value_counts())


Number of examples: 5427

Class distribution:
emotion
27         1606
0           348
15          260
4           236
10          195
           ... 
4,5,18        1
7,13,26       1
0,1,22        1
1,20,27       1
10,12         1
Name: count, Length: 275, dtype: int64


In [16]:
valid_df = pd.read_csv(VALID_DATA_PATH, sep='\t', header=None, names=["text", "emotion", "userid"])

# Remove rows with missing text/labels
valid_df = valid_df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])

print("Number of examples:", len(valid_df))
print("\nClass distribution:")
print(valid_df[LABEL_COLUMN].value_counts())


Number of examples: 5426

Class distribution:
emotion
27            1592
0              326
15             261
4              258
10             212
              ... 
5,14             1
5,8,20,25        1
0,4,9            1
0,13,15,26       1
0,17,22          1
Name: count, Length: 296, dtype: int64


In [17]:
# ============================================================
# 4. Encode labels
# ============================================================

with open(EMOTIONS_PATH, encoding="utf-8") as emotions_file:
    labels = [line.strip() for line in emotions_file if line.strip()]

with open(EKMAN_PATH, encoding="utf-8") as ekman_file:
    ekman_mapping = json.load(ekman_file)

label2id = {label: index for index, label in enumerate(labels)}
id2label = {index: label for label, index in label2id.items()}

category_labels = list(ekman_mapping)
category_labels.append("neutral")
category2id = {category: index for index, category in enumerate(category_labels)}
id2category = {index: category for category, index in category2id.items()}

emotion_to_category_id = {}
for category, emotions in ekman_mapping.items():
    for emotion in emotions:
        emotion_to_category_id[label2id[emotion]] = category2id[category]

emotion_to_category_id[label2id["neutral"]] = category2id["neutral"]
NUM_LABELS = len(category_labels)

def encode_category(emotion_indices):
    first_emotion_index = int(str(emotion_indices).split(",")[0])
    return emotion_to_category_id[first_emotion_index]

for df in [train_df, valid_df, test_df]:
    df["category"] = df[LABEL_COLUMN].apply(encode_category)

print("\nCategory labels:")
print(category2id)




Category labels:
{'anger': 0, 'disgust': 1, 'fear': 2, 'joy': 3, 'sadness': 4, 'surprise': 5, 'neutral': 6}


In [18]:
"""
# ============================================================
# 5. Train / validation / test split
# ============================================================

# First separate out the test set
train_val_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=df["category"]
)

# Then split the remaining 80% into
# 60% train and 20% validation
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=train_val_df["category"]
)
"""
print("\nDataset sizes:")
print("Train:", len(train_df))
print("Validation:", len(valid_df))
print("Test:", len(test_df))




Dataset sizes:
Train: 43410
Validation: 5426
Test: 5427


In [19]:

# ============================================================
# 6. Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, timeout=120)



In [20]:
# ============================================================
# 7. PyTorch Dataset
# ============================================================

class EmotionDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe[TEXT_COLUMN].tolist()
        self.labels = dataframe["category"].astype(int).tolist()

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        text = str(self.texts[index])
        label = self.labels[index]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }


train_dataset = EmotionDataset(
    train_df,
    tokenizer,
    MAX_LENGTH
)

val_dataset = EmotionDataset(
    valid_df,
    tokenizer,
    MAX_LENGTH
)

test_dataset = EmotionDataset(
    test_df,
    tokenizer,
    MAX_LENGTH
)



In [21]:

# ============================================================
# 8. DataLoaders
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)



In [22]:
# ============================================================
# 9. Model
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2category,
    label2id=category2id,
    problem_type="single_label_classification"
)

model.to(DEVICE)



Loading weights: 100%|██████████| 197/197 [00:00<00:00, 21182.81it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=768, out_features=7, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Li

In [23]:

# ============================================================
# 10. Optimizer and scheduler
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

total_training_steps = (
    len(train_loader) * EPOCHS
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_training_steps),
    num_training_steps=total_training_steps
)



In [24]:
# ============================================================
# 11. Evaluation function
# ============================================================

def evaluate(model, dataloader):

    model.eval()

    all_predictions = []
    all_labels = []

    total_loss = 0

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels_batch = batch["labels"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels_batch
            )

            total_loss += outputs.loss.item()
            predictions = outputs.logits.argmax(dim=-1)

            all_predictions.extend(predictions.cpu().tolist())
            all_labels.extend(labels_batch.cpu().tolist())

    average_loss = total_loss / len(dataloader)

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "loss": average_loss,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "labels": all_labels,
        "predictions": all_predictions
    }



In [25]:

# ============================================================
# 12. Training
# ============================================================

best_val_f1 = -float("inf")
best_model_path = "best_e0_model.pt"

for epoch in range(EPOCHS):

    model.train()

    total_train_loss = 0

    for batch in train_loader:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels_batch = batch["labels"].to(DEVICE)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_batch
        )

        loss = outputs.loss

        loss.backward()

        # Prevent excessively large gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

    average_train_loss = (
        total_train_loss / len(train_loader)
    )

    # Validation
    val_results = evaluate(
        model,
        val_loader
    )

    print(
        f"\nEpoch {epoch + 1}/{EPOCHS}"
    )

    print(
        f"Train loss: "
        f"{average_train_loss:.4f}"
    )

    print(
        f"Validation loss: "
        f"{val_results['loss']:.4f}"
    )

    print(
        f"Validation accuracy: "
        f"{val_results['accuracy']:.4f}"
    )

    print(
        f"Validation macro-F1: "
        f"{val_results['macro_f1']:.4f}"
    )

    print(
        f"Validation weighted-F1: "
        f"{val_results['weighted_f1']:.4f}"
    )

    # Save best model based on macro-F1
    if val_results["macro_f1"] > best_val_f1:

        best_val_f1 = val_results["macro_f1"]

        torch.save(
            model.state_dict(),
            best_model_path
        )

        print("Saved new best model.")




Epoch 1/5
Train loss: 1.1269
Validation loss: 0.8682
Validation accuracy: 0.6858
Validation macro-F1: 0.5566
Validation weighted-F1: 0.6799
Saved new best model.

Epoch 2/5
Train loss: 0.8331
Validation loss: 0.8355
Validation accuracy: 0.6913
Validation macro-F1: 0.6015
Validation weighted-F1: 0.6885
Saved new best model.

Epoch 3/5
Train loss: 0.7149
Validation loss: 0.8784
Validation accuracy: 0.6893
Validation macro-F1: 0.6127
Validation weighted-F1: 0.6853
Saved new best model.

Epoch 4/5
Train loss: 0.6017
Validation loss: 0.9319
Validation accuracy: 0.6919
Validation macro-F1: 0.6186
Validation weighted-F1: 0.6876
Saved new best model.

Epoch 5/5
Train loss: 0.5086
Validation loss: 0.9917
Validation accuracy: 0.6828
Validation macro-F1: 0.6088
Validation weighted-F1: 0.6790


In [26]:

# ============================================================
# 13. Load best model
# ============================================================

model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=DEVICE
    )
)



/tmp/ipykernel_1364540/2246141660.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


<All keys matched successfully>

In [27]:

# ============================================================
# 14. Final test evaluation
# ============================================================

test_results = evaluate(
    model,
    test_loader
)

print("\n==============================")
print("FINAL E0 TEST RESULTS")
print("==============================")

print(
    f"Accuracy:     "
    f"{test_results['accuracy']:.4f}"
)

print(
    f"Macro-F1:     "
    f"{test_results['macro_f1']:.4f}"
)

print(
    f"Weighted-F1:  "
    f"{test_results['weighted_f1']:.4f}"
)




FINAL E0 TEST RESULTS
Accuracy:     0.6792
Macro-F1:     0.6032
Weighted-F1:  0.6743


In [28]:
# ============================================================
# 15. Per-category results
# ============================================================

print("\nClassification Report:")

print(
    classification_report(
        test_results["labels"],
        test_results["predictions"],
        labels=list(range(NUM_LABELS)),
        target_names=category_labels,
        digits=4,
        zero_division=0
    )
)




Classification Report:
              precision    recall  f1-score   support

       anger     0.5538    0.5633    0.5585       703
     disgust     0.4868    0.4405    0.4625        84
        fear     0.6344    0.6556    0.6448        90
         joy     0.7746    0.8700    0.8195      2054
     sadness     0.5753    0.5300    0.5517       317
    surprise     0.5609    0.5864    0.5734       573
     neutral     0.6714    0.5623    0.6120      1606

    accuracy                         0.6792      5427
   macro avg     0.6082    0.6011    0.6032      5427
weighted avg     0.6745    0.6792    0.6743      5427



In [29]:
# ============================================================
# 16. Confusion matrix
# ============================================================

category_confusion_matrix = confusion_matrix(
    test_results["labels"],
    test_results["predictions"],
    labels=list(range(NUM_LABELS))
)

print("\nCategory Confusion Matrix:")
print(
    pd.DataFrame(
        category_confusion_matrix,
        index=category_labels,
        columns=category_labels
    )
)




Category Confusion Matrix:
          anger  disgust  fear   joy  sadness  surprise  neutral
anger       396       19     6    71       31        39      141
disgust      23       37     3     8        4         4        5
fear          7        3    59     3        8         3        7
joy          47        3    11  1787       16        36      154
sadness      30        3     3    39      168        24       50
surprise     31        3     4    93       21       336       85
neutral     181        8     7   306       44       157      903
